In [15]:
import os
import glob
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import multilabel_confusion_matrix
import matplotlib.pyplot as plt

In [16]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 25
LR = 1e-4

In [17]:
DATA_ROOT = "/kaggle/input/data"

image_paths = {}
for folder in glob.glob(f"{DATA_ROOT}/images_*/images"):
    for img_path in glob.glob(f"{folder}/*.png"):
        image_paths[os.path.basename(img_path)] = img_path

In [18]:
df = pd.read_csv(f"{DATA_ROOT}/Data_Entry_2017.csv")
bbox_df = pd.read_csv(f"{DATA_ROOT}/BBox_List_2017.csv")

In [19]:
CLASSES = [
    "Atelectasis",
    "Cardiomegaly",
    "Effusion",
    "Infiltration",
    "Mass",
    "Nodule",
    "Pneumonia",
    "Pneumothorax",
    "Consolidation",
    "No Finding"
]

In [20]:
for c in CLASSES:
    df[c] = df["Finding Labels"].apply(lambda x: 1 if c in x else 0)

# Keep only rows with at least one of the 10 classes
df = df[df[CLASSES].sum(axis=1) > 0]

# Keep only images that exist
df = df[df["Image Index"].isin(image_paths.keys())]

In [21]:
train_df, val_df = train_test_split(
    df, test_size=0.2, random_state=42
)

In [22]:
class NIHChestDataset(Dataset):
    def __init__(self, dataframe, bbox_df, image_paths, transform):
        self.df = dataframe.reset_index(drop=True)
        self.bbox_df = bbox_df
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.loc[idx]
        img_path = self.image_paths[row["Image Index"]]

        image = Image.open(img_path).convert("RGB")
        image = self.transform(image)

        labels = torch.tensor(
            row[CLASSES].values.astype(np.float32)
        )

        # BBox mask
        bbox_mask = torch.zeros((IMG_SIZE, IMG_SIZE))
        bboxes = self.bbox_df[self.bbox_df["Image Index"] == row["Image Index"]]

        for _, b in bboxes.iterrows():
            x = int(b["Bbox [x"])
            y = int(b["y"])
            w = int(b["w"])
            h = int(b["h]"])
            bbox_mask[y:y+h, x:x+w] = 1.0

        return image, labels, bbox_mask.unsqueeze(0)

In [23]:
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

In [24]:
train_ds = NIHChestDataset(train_df, bbox_df, image_paths, train_tf)
val_ds   = NIHChestDataset(val_df, bbox_df, image_paths, val_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [25]:
class EfficientNetXAI(nn.Module):
    def __init__(self):
        super().__init__()
        base = models.efficientnet_b0(weights="IMAGENET1K_V1")
        self.features = base.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(1280, len(CLASSES))

    def forward(self, x):
        feat = self.features(x)
        pooled = self.pool(feat).flatten(1)
        out = self.classifier(pooled)
        return out, feat

In [26]:
model = EfficientNetXAI().to(DEVICE)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 133MB/s] 


In [28]:
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    patience=3,
    factor=0.3
)

In [29]:
def bbox_attention_loss(feat_map, bbox):
    attn = feat_map.mean(dim=1, keepdim=True)
    attn = F.interpolate(attn, size=(IMG_SIZE, IMG_SIZE), mode="bilinear")
    return F.mse_loss(attn, bbox)

In [30]:
best_val_loss = 1e9
patience = 5
counter = 0

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0

    for imgs, labels, bbox in tqdm(train_loader):
        imgs = imgs.to(DEVICE)
        labels = labels.to(DEVICE)
        bbox = bbox.to(DEVICE)

        optimizer.zero_grad()
        outputs, feats = model(imgs)

        loss = criterion(outputs, labels)
        loss += 0.2 * bbox_attention_loss(feats, bbox)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # Validation
    model.eval()
    val_loss, correct, total = 0, 0, 0

    with torch.no_grad():
        for imgs, labels, _ in val_loader:
            imgs = imgs.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs, _ = model(imgs)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            preds = torch.sigmoid(outputs) > 0.5
            correct += (preds == labels.bool()).sum().item()
            total += labels.numel()

    val_acc = correct / total
    scheduler.step(val_loss)

    print(f"\nEpoch {epoch+1}")
    print(f"Train Loss : {train_loss/len(train_loader):.4f}")
    print(f"Val Loss   : {val_loss/len(val_loader):.4f}")
    print(f"Val Acc    : {val_acc:.4f}")
    print(f"LR         : {optimizer.param_groups[0]['lr']}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_model.pth")
        counter = 0
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping triggered")
            break

100%|██████████| 5424/5424 [23:27<00:00,  3.85it/s]



Epoch 1
Train Loss : 0.2540
Val Loss   : 0.2374
Val Acc    : 0.9071
LR         : 0.0001


100%|██████████| 5424/5424 [23:21<00:00,  3.87it/s]



Epoch 2
Train Loss : 0.2336
Val Loss   : 0.2330
Val Acc    : 0.9087
LR         : 0.0001


100%|██████████| 5424/5424 [22:28<00:00,  4.02it/s]


...


Epoch 6
Train Loss : 0.2057
Val Loss   : 0.2376
Val Acc    : 0.9065
LR         : 0.0001


100%|██████████| 5424/5424 [22:17<00:00,  4.06it/s]



Epoch 7
Train Loss : 0.1970
Val Loss   : 0.2441
Val Acc    : 0.9040
LR         : 3e-05


100%|██████████| 5424/5424 [21:54<00:00,  4.13it/s]



Epoch 8
Train Loss : 0.1764
Val Loss   : 0.2565
Val Acc    : 0.9036
LR         : 3e-05
Early stopping triggered


In [31]:
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

all_preds, all_labels = [], []

with torch.no_grad():
    for imgs, labels, _ in val_loader:
        imgs = imgs.to(DEVICE)
        outputs, _ = model(imgs)
        preds = (torch.sigmoid(outputs) > 0.5).cpu()
        all_preds.append(preds)
        all_labels.append(labels)

all_preds = torch.cat(all_preds)
all_labels = torch.cat(all_labels)

mcm = multilabel_confusion_matrix(all_labels, all_preds)

In [32]:
class GradCAM:
    def __init__(self, model):
        self.model = model
        self.grad = None
        self.act = None

        self.model.features[-1].register_forward_hook(self._save_act)
        self.model.features[-1].register_backward_hook(self._save_grad)

    def _save_act(self, m, i, o):
        self.act = o

    def _save_grad(self, m, gi, go):
        self.grad = go[0]

    def generate(self, x, cls_idx):
        self.model.zero_grad()
        out, _ = self.model(x)
        out[:, cls_idx].backward()

        weights = self.grad.mean(dim=(2,3), keepdim=True)
        cam = (weights * self.act).sum(dim=1)
        cam = F.relu(cam)
        cam = cam / cam.max()
        return cam

In [33]:
img = Image.open(list(image_paths.values())[0]).convert("RGB")
img_t = val_tf(img).unsqueeze(0).to(DEVICE)

cam_gen = GradCAM(model)
cam = cam_gen.generate(img_t, cls_idx=0)

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:1864: FutureWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  self._maybe_warn_non_full_backward_hook(args, result, grad_fn)
